In [1]:
import anndata as ad
import numpy as np
import pandas as pd

print("AnnData version:", ad.__version__)
print("NumPy version:", np.__version__)
print("All imports successful!")

AnnData version: 0.11.4
NumPy version: 2.2.6
All imports successful!


In [2]:
# Create a simulated count matrix: 100 cells × 2000 genes
np.random.seed(42)
n_obs, n_vars = 100, 2000

X = np.random.poisson(lam=5, size=(n_obs, n_vars)).astype(float)

# Create AnnData object
adata = ad.AnnData(X=X)

# Assign cell and gene names
adata.obs_names = [f'Cell_{i}' for i in range(n_obs)]
adata.var_names = [f'Gene_{i}' for i in range(n_vars)]

print(adata)

AnnData object with n_obs × n_vars = 100 × 2000


In [3]:
# Add cell-level metadata to .obs
adata.obs['cell_type'] = pd.Categorical(
    np.random.choice(['B', 'T', 'Monocyte'], size=n_obs)
)
adata.obs['batch'] = pd.Categorical(
    np.random.choice(['batch1', 'batch2'], size=n_obs)
)
adata.obs['n_counts'] = X.sum(axis=1)

# Add gene-level metadata to .var
adata.var['gene_name'] = [f'Gene_{i}' for i in range(n_vars)]
adata.var['highly_variable'] = np.random.choice([True, False], size=n_vars)

print(adata)
print("\n--- Cell metadata (first 5 rows) ---")
print(adata.obs.head())

AnnData object with n_obs × n_vars = 100 × 2000
    obs: 'cell_type', 'batch', 'n_counts'
    var: 'gene_name', 'highly_variable'

--- Cell metadata (first 5 rows) ---
       cell_type   batch  n_counts
Cell_0  Monocyte  batch1    9901.0
Cell_1  Monocyte  batch1   10234.0
Cell_2         B  batch1    9924.0
Cell_3         T  batch1    9955.0
Cell_4         T  batch1    9991.0


In [4]:
# Add 2D UMAP embedding to .obsm
adata.obsm['X_umap'] = np.random.normal(0, 1, size=(n_obs, 2))

# Add 50D PCA embedding to .obsm
adata.obsm['X_pca'] = np.random.normal(0, 1, size=(n_obs, 50))

# Add gene-level metadata to .varm
adata.varm['gene_stuff'] = np.random.normal(0, 1, size=(n_vars, 5))

print(adata)
print("\n--- UMAP embedding shape ---")
print(adata.obsm['X_umap'].shape)

AnnData object with n_obs × n_vars = 100 × 2000
    obs: 'cell_type', 'batch', 'n_counts'
    var: 'gene_name', 'highly_variable'
    obsm: 'X_umap', 'X_pca'
    varm: 'gene_stuff'

--- UMAP embedding shape ---
(100, 2)


In [5]:
# Add unstructured metadata to .uns
adata.uns['experiment'] = {
    'organism': 'Homo sapiens',
    'tissue': 'PBMC',
    'date': '2026-04-01',
    'instrument': '10X Chromium v3'
}

# Add layers (alternative count matrices)
adata.layers['raw_counts'] = adata.X.copy()
adata.layers['log_transformed'] = np.log1p(adata.X)

print(adata)
print("\n--- Experiment metadata ---")
print(adata.uns['experiment'])
print("\n--- Log transformed (first 3 cells, first 5 genes) ---")
print(adata.layers['log_transformed'][:3, :5])

AnnData object with n_obs × n_vars = 100 × 2000
    obs: 'cell_type', 'batch', 'n_counts'
    var: 'gene_name', 'highly_variable'
    uns: 'experiment'
    obsm: 'X_umap', 'X_pca'
    varm: 'gene_stuff'
    layers: 'raw_counts', 'log_transformed'

--- Experiment metadata ---
{'organism': 'Homo sapiens', 'tissue': 'PBMC', 'date': '2026-04-01', 'instrument': '10X Chromium v3'}

--- Log transformed (first 3 cells, first 5 genes) ---
[[1.79175947 1.60943791 1.60943791 1.79175947 1.79175947]
 [2.19722458 2.48490665 1.60943791 1.09861229 1.38629436]
 [2.19722458 1.38629436 1.38629436 1.79175947 1.79175947]]


In [6]:
# Subset by cell type (only B cells)
b_cells = adata[adata.obs['cell_type'] == 'B']
print("B cells only:", b_cells)

# Subset by highly variable genes
hvg = adata[:, adata.var['highly_variable']]
print("\nHighly variable genes only:", hvg)

# Combined subset: B cells + HVGs
subset = adata[adata.obs['cell_type'] == 'B', adata.var['highly_variable']]
print("\nB cells + HVGs:", subset)

B cells only: View of AnnData object with n_obs × n_vars = 34 × 2000
    obs: 'cell_type', 'batch', 'n_counts'
    var: 'gene_name', 'highly_variable'
    uns: 'experiment'
    obsm: 'X_umap', 'X_pca'
    varm: 'gene_stuff'
    layers: 'raw_counts', 'log_transformed'

Highly variable genes only: View of AnnData object with n_obs × n_vars = 100 × 1027
    obs: 'cell_type', 'batch', 'n_counts'
    var: 'gene_name', 'highly_variable'
    uns: 'experiment'
    obsm: 'X_umap', 'X_pca'
    varm: 'gene_stuff'
    layers: 'raw_counts', 'log_transformed'

B cells + HVGs: View of AnnData object with n_obs × n_vars = 34 × 1027
    obs: 'cell_type', 'batch', 'n_counts'
    var: 'gene_name', 'highly_variable'
    uns: 'experiment'
    obsm: 'X_umap', 'X_pca'
    varm: 'gene_stuff'
    layers: 'raw_counts', 'log_transformed'


In [7]:
import os
os.makedirs('outputs', exist_ok=True)
os.makedirs('outputs/tutorial_outputs', exist_ok=True)

# Save full AnnData to disk
adata.write('outputs/sample_anndata.h5ad')
print("Saved successfully!")

# Save subset
subset_copy = subset.copy()
subset_copy.write('outputs/tutorial_outputs/subset_adata.h5ad')
print("Subset saved!")

# Reload and verify
adata_loaded = ad.read_h5ad('outputs/sample_anndata.h5ad')
print("\n--- Reloaded AnnData ---")
print(adata_loaded)
print("\nAll components preserved!")

Saved successfully!
Subset saved!

--- Reloaded AnnData ---
AnnData object with n_obs × n_vars = 100 × 2000
    obs: 'cell_type', 'batch', 'n_counts'
    var: 'gene_name', 'highly_variable'
    uns: 'experiment'
    obsm: 'X_pca', 'X_umap'
    varm: 'gene_stuff'
    layers: 'log_transformed', 'raw_counts'

All components preserved!


In [8]:
# Views vs Copies
view = adata[:50]
print("Is view:", view.is_view)

copy = adata[:50].copy()
print("Is copy:", copy.is_view)
copy.obs['new_col'] = 'test'
print("Modified copy successfully!")

# Concatenate two AnnData objects
adata_b2 = adata.copy()
adata_b2.obs_names = [f'Cell_b2_{i}' for i in range(adata_b2.n_obs)]

adata_combined = ad.concat(
    [adata, adata_b2],
    keys=['batch1', 'batch2'],
    label='batch',
    join='outer',
    fill_value=0
)

print("\n--- Combined AnnData ---")
print(adata_combined)

# Save
adata_combined.write('outputs/tutorial_outputs/concatenated_adata.h5ad')
print("\nConcatenated AnnData saved!")


Is view: True
Is copy: False
Modified copy successfully!

--- Combined AnnData ---
AnnData object with n_obs × n_vars = 200 × 2000
    obs: 'cell_type', 'batch', 'n_counts'
    obsm: 'X_umap', 'X_pca'
    layers: 'raw_counts', 'log_transformed'

Concatenated AnnData saved!
